# PS-OCT Processing Toolbox Tutorial

This notebook demonstrates a first complete workflow for the `psoct-processing` Python toolbox. It uses the synthetic data generator included with the package, so it can be run before private PS-OCT acquisitions are available. The synthetic data are an engineering phantom for testing I/O, reconstruction, export, and validation; they are not a biophysical PS-OCT simulation.

The tutorial covers installation checks, configuration, synthetic data generation, command-line use, direct Python API use, output inspection, visualization, and validation scaffolding.


## 1. Installation

From the repository root, install the package in editable mode. The notebook assumes that it is being run from the repository root or that the package has already been installed in the active Python environment.

```bash
pip install -e .
```

For development and testing, use:

```bash
pip install -e .[dev]
pytest
```


In [ ]:
from pathlib import Path
import sys

# Make the tutorial robust when run directly from the notebooks/ folder.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt

import psoct_processing
from psoct_processing.synthetic import write_synthetic_dat, synthetic_config
from psoct_processing.pipeline import run_pipeline
from psoct_processing.config import load_config

print("Repository root:", repo_root)
print("psoct_processing version:", getattr(psoct_processing, "__version__", "unknown"))


## 2. Create a synthetic PS-OCT-like acquisition

The generator creates a deterministic multi-channel spectral cube and writes it as a raw `.dat` file. The default synthetic layout is channel-first, with shape `[channel, spectral_sample, x, y]`. This allows the pipeline to be exercised without committing to a final real-instrument acquisition layout.


In [ ]:
work_dir = repo_root / "tutorial_outputs"
raw_dir = work_dir / "synthetic_raw"
processed_dir = work_dir / "processed"

products = write_synthetic_dat(
    raw_dir,
    filename="synthetic_psoct.dat",
    n_channels=2,
    spectral_samples=512,
    x=32,
    y=16,
    seed=7,
)

products


## 3. Create and save a configuration file

For real acquisitions, the configuration file is where acquisition dimensions, channel order, reconstruction parameters, export options, and provenance settings should be made explicit. In this tutorial, the configuration is generated programmatically and saved as YAML.


In [ ]:
import yaml

cfg = synthetic_config(n_channels=2, spectral_samples=512, x=32, y=16)
cfg.export.save_intermediates = True
cfg.export.save_tiff = True
cfg.export.save_zarr = True

config_path = work_dir / "synthetic_config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(cfg.model_dump(mode="json"), f, sort_keys=False)

print(config_path)


## 4. Run the processing pipeline through the Python API

The pipeline discovers raw files, applies background subtraction, reconstructs the complex volume, computes the primary channel reflectivity, creates an en-face projection, computes two-channel polarization diagnostics when `co_pol` and `cross_pol` are present, and writes provenance metadata.


In [ ]:
loaded_cfg = load_config(config_path)
results = run_pipeline(raw_dir, processed_dir, loaded_cfg)

print(f"Processed {len(results)} file(s)")
print(results[0]["products"].keys())


## 5. Inspect generated outputs

The current engineering version writes TIFF products for visualization, Zarr products for array-oriented downstream processing, and JSON metadata/provenance files. Depending on whether `zarr` is installed in the environment, the development fallback may store NumPy arrays inside the `.zarr` directory.


In [ ]:
for path in sorted(processed_dir.glob("*")):
    print(path.relative_to(repo_root))


## 6. Load and visualize a TIFF product

The reflectivity TIFF is a 3D array with depth and spatial dimensions. The en-face TIFF is a 2D projection generated from a depth range or the full reconstructed depth axis.


In [ ]:
import tifffile

enface_path = processed_dir / "synthetic_psoct_co_pol_enface.tiff"
reflectivity_path = processed_dir / "synthetic_psoct_co_pol_reflectivity.tiff"

enface = tifffile.imread(enface_path)
reflectivity = tifffile.imread(reflectivity_path)

print("enface shape:", enface.shape)
print("reflectivity shape:", reflectivity.shape)

plt.figure(figsize=(6, 4))
plt.imshow(enface, aspect="auto")
plt.title("Synthetic co-pol en-face projection")
plt.xlabel("y")
plt.ylabel("x")
plt.colorbar(label="a.u.")
plt.tight_layout()
plt.show()


## 7. Visualize a reconstructed depth slice

This slice is useful for checking whether the reconstruction stage is producing coherent structure before moving to real PS-OCT acquisitions.


In [ ]:
depth_index = reflectivity.shape[0] // 4

plt.figure(figsize=(6, 4))
plt.imshow(reflectivity[depth_index], aspect="auto")
plt.title(f"Reflectivity slice at depth index {depth_index}")
plt.xlabel("y")
plt.ylabel("x")
plt.colorbar(label="dB")
plt.tight_layout()
plt.show()


## 8. Use the command-line interface

The same operations can be run from the terminal. The package exposes a single Typer application named `psoct-process`, with subcommands for initialization, synthetic data generation, processing, inspection, and validation.


In [ ]:
# These commands are shown for reference. They can be copied into a terminal.
print(f"psoct-process generate-synthetic --output-dir {raw_dir} --spectral-samples 512 --x 32 --y 16 --n-channels 2")
print(f"psoct-process run --input-dir {raw_dir} --output-dir {processed_dir} --config {config_path}")
print(f"psoct-process inspect --input-dir {raw_dir} --config {config_path}")


## 9. Validation workflow scaffold

Once MATLAB reference outputs are available, the validation harness can compare matching intermediate arrays from MATLAB and Python. A future real-data validation run should export intermediate MATLAB arrays for stages such as background-corrected spectra, interpolated spectra, complex reconstruction, reflectivity, retardance, en-face projection, and stitched mosaics.

The command-line interface will then compare corresponding files by stage name and report RMSE, relative RMSE, absolute error, and Pearson correlation.


In [ ]:
print("Example validation command:")
print("psoct-process validate --matlab-output ./matlab_reference --python-output ./python_output --report validation_report.json")


## 10. What changes for real data?

For real acquisitions, the main work is to replace the synthetic configuration with the true acquisition schema. The most important fields are the raw array shape, byte order, spectral axis, channel layout, channel names, acquisition-specific header behavior, interpolation/calibration files, and dispersion compensation parameters.

The recommended first real-data validation experiment is intentionally small: one raw tile, one MATLAB reconstruction, and MATLAB intermediate arrays saved after each major stage. Once the Python pipeline matches the MATLAB pathway qualitatively and then quantitatively for this small case, the same configuration can be scaled to larger mosaics and whole-volume processing.
